# 00 - Preparación de bases EPH

Este notebook descarga las bases de la EPH (vía `pyeph`, y bases manuales subidas a `data/raw/` para los trimestres que `pyeph` todavía no tenga), une la base de **individuos** con la de **hogares** por el código de vínculo (`CODUSU` + `NRO_HOGAR`), y deja los datasets listos en `data/processed/` para que los usen el resto de los notebooks.

**Salidas:**
- `data/processed/eph_T<Q><YY>.parquet`: un archivo por trimestre, ya con individuos+hogares unidos.
- `data/processed/eph_panel.parquet`: panel con todos los trimestres concatenados (con columnas `ANIO` y `TRIMESTRE`).

**Cómo cargar un trimestre nuevo que pyeph todavía no tiene:**
1. Descargá del INDEC los archivos `usu_individual_TxQyy.txt` y `usu_hogar_TxQyy.txt` (o `.xls`) del trimestre.
2. Subilos a `data/raw/` con el nombre `usu_individual_T<Q><YY>.txt` y `usu_hogar_T<Q><YY>.txt` (ej: `usu_individual_T424.txt`, `usu_hogar_T424.txt` para T4 2024).
3. Agregá el trimestre a la lista `QUARTERS` de abajo y volvé a correr el notebook.

## Setup (Colab)

Clona el repo para tener acceso a `src/data_loader.py` y a `data/raw/`, e instala `pyeph`.

In [ ]:
import sys, os

REPO_URL = "https://github.com/santiagoriverti/analisis_EPH.git"
REPO_DIR = "analisis_EPH"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL}

%cd {REPO_DIR}
!pip install -q pyeph

sys.path.append(os.getcwd())

## Trimestres a incluir en el panel

Editar esta lista a medida que haya nuevos trimestres disponibles (en `pyeph` o subidos a `data/raw/`).

In [ ]:
QUARTERS = [
    (2023, 1), (2023, 2), (2023, 3), (2023, 4),
    (2024, 1), (2024, 2), (2024, 3), (2024, 4),
]

## Chequeo de disponibilidad

Para cada trimestre, intenta `pyeph` y si falla busca en `data/raw/`. Reporta qué trimestres no se pudieron resolver de ninguna forma (para que los subas manualmente).

In [ ]:
from src.data_loader import load_eph, _period_tag

available, missing = [], []

for year, period in QUARTERS:
    try:
        load_eph(year, period, base_type="individual")
        load_eph(year, period, base_type="hogar")
        available.append((year, period))
    except FileNotFoundError:
        missing.append((year, period))

print("Disponibles:", [_period_tag(y, p) for y, p in available])
print("Faltantes (subir a data/raw/):", [_period_tag(y, p) for y, p in missing])

## Construcción del panel

Une individuos + hogares por trimestre (`CODUSU` + `NRO_HOGAR`), agrega `ANIO`/`TRIMESTRE`, y guarda en `data/processed/`.

In [ ]:
from src.data_loader import build_panel

panel = build_panel(available, save=True)
panel.shape

In [ ]:
panel[["ANIO", "TRIMESTRE", "CODUSU", "NRO_HOGAR", "COMPONENTE"]].head()

## Subir resultados a GitHub (opcional)

Los archivos `.parquet` generados en `data/processed/` quedan disponibles para los demás notebooks vía `raw.githubusercontent.com` una vez que se comiteen y pusheen al repo (esto se hace desde el entorno local, no desde Colab).